In [1]:
!pip install -q pymupdf

import os
import requests
import fitz

os.makedirs("/content/task2_rag", exist_ok=True)

pdf_url = "https://nvlpubs.nist.gov/nistpubs/ai/NIST.AI.100-1.pdf"

pdf_path = "/content/task2_rag/nist_ai_rmf.pdf"

response = requests.get(pdf_url)
response.raise_for_status()

with open(pdf_path, "wb") as f:
    f.write(response.content)

doc = fitz.open(pdf_path)

print("PDF downloaded successfully!")
print("Document:", pdf_path)
print("Total pages:", len(doc))
print("File size:", round(os.path.getsize(pdf_path) / (1024*1024), 2), "MB")

/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


PDF downloaded successfully!
Document: /content/task2_rag/nist_ai_rmf.pdf
Total pages: 48
File size: 1.86 MB


In [2]:
import pymupdf
import os

pdf_path = "/content/task2_rag/nist_ai_rmf.pdf"

doc = pymupdf.open(pdf_path)

pages = []

for page_num, page in enumerate(doc):
    text = page.get_text("text")

    if text.strip():
        pages.append({
            "page": page_num + 1,
            "text": text.strip()
        })

print("Total PDF pages:", len(doc))
print("Pages with extracted text:", len(pages))

print("\nFirst page preview:")
print(pages[0]["text"][:1500])

Total PDF pages: 48
Pages with extracted text: 48

First page preview:
NIST AI 100-1
Artificial Intelligence Risk Management
Framework (AI RMF 1.0)


In [3]:
def create_chunks(pages, chunk_size, overlap=100):
    chunks = []

    for page in pages:
        text = page["text"]
        start = 0

        while start < len(text):
            end = min(start + chunk_size, len(text))

            # Prefer ending at a word boundary
            if end < len(text):
                boundary = text.rfind(" ", start, end)
                if boundary > start:
                    end = boundary

            chunk_text = text[start:end].strip()

            if chunk_text:
                chunks.append({
                    "text": chunk_text,
                    "page": page["page"],
                    "chunk_size": chunk_size
                })

            if end >= len(text):
                break

            start = max(start + 1, end - overlap)

    return chunks


# Configuration A: 500 characters
chunks_500 = create_chunks(
    pages,
    chunk_size=500,
    overlap=100
)

# Configuration B: 1000 characters
chunks_1000 = create_chunks(
    pages,
    chunk_size=1000,
    overlap=100
)

print("CONFIGURATION A")
print("Chunk size: 500")
print("Total chunks:", len(chunks_500))

print("\nCONFIGURATION B")
print("Chunk size: 1000")
print("Total chunks:", len(chunks_1000))

print("\nSample chunk:")
print(chunks_500[0])

CONFIGURATION A
Chunk size: 500
Total chunks: 278

CONFIGURATION B
Chunk size: 1000
Total chunks: 137

Sample chunk:
{'text': 'NIST AI 100-1\nArtificial Intelligence Risk Management\nFramework (AI RMF 1.0)', 'page': 1, 'chunk_size': 500}


In [4]:
!pip install -q sentence-transformers faiss-cpu

from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Load embedding model
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

def build_faiss_index(chunks):
    texts = [chunk["text"] for chunk in chunks]

    embeddings = embedding_model.encode(
        texts,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True
    )

    embeddings = embeddings.astype("float32")

    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)

    return index, embeddings

# Build index for Configuration A
index_500, embeddings_500 = build_faiss_index(chunks_500)

# Build index for Configuration B
index_1000, embeddings_1000 = build_faiss_index(chunks_1000)

print("\nEmbedding model loaded successfully!")
print("Embedding dimensions:", embeddings_500.shape[1])

print("\nConfiguration A:")
print("Indexed vectors:", index_500.ntotal)

print("\nConfiguration B:")
print("Indexed vectors:", index_1000.ntotal)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (6 outputs left out)

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/9 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]


Embedding model loaded successfully!
Embedding dimensions: 384

Configuration A:
Indexed vectors: 278

Configuration B:
Indexed vectors: 137


In [5]:
# Find relevant passages and their original PDF page numbers

search_topics = [
    "AI Risk Management Framework",
    "trustworthy AI",
    "valid and reliable",
    "safe",
    "secure and resilient",
    "accountable and transparent",
    "explainable and interpretable",
    "privacy-enhanced",
    "harmful bias",
    "Govern",
    "Map",
    "Measure",
    "Manage",
    "risk tolerance",
    "AI actors"
]

for topic in search_topics:
    print("\n" + "=" * 65)
    print("TOPIC:", topic)

    found = False

    for page in pages:
        text = page["text"]
        position = text.lower().find(topic.lower())

        if position != -1:
            start = max(0, position - 180)
            end = min(len(text), position + len(topic) + 300)

            print("PDF Page:", page["page"])
            print(text[start:end].replace("\n", " "))
            found = True
            break

    if not found:
        print("No direct match found.")


TOPIC: AI Risk Management Framework
PDF Page: 8
 research and development and evaluation conducted by NIST and others. Priority research and additional guidance that will enhance this Framework will be captured in an associated AI Risk Management Framework Roadmap to which NIST and the broader community can contribute. Page 3

TOPIC: trustworthy AI
PDF Page: 5
NIST AI 100-1 AI RMF 1.0 List of Figures Fig. 1 Examples of potential harms related to AI systems. Trustworthy AI systems and their responsible use can mitigate negative risks and contribute to bene- fits for people, organizations, and ecosystems. 5 Fig. 2 Lifecycle and Key Dimensions of an AI System. Modified from OECD (2022) OECD Framework for the Classification of AI systems — OECD Digital Economy Papers. T

TOPIC: valid and reliable
PDF Page: 4
 Risk Measurement 5 1.2.2 Risk Tolerance 7 1.2.3 Risk Prioritization 7 1.2.4 Organizational Integration and Management of Risk 8 2 Audience 9 3 AI Risks and Trustworthiness 12 3.1 Val

In [7]:
import re
print("Regular expression module imported successfully!")
topics = [
    "valid and reliable",
    "safe",
    "secure and resilient",
    "accountable and transparent",
    "explainable and interpretable",
    "privacy-enhanced",
    "harmful bias",
    "Govern",
    "Map",
    "Measure",
    "Manage",
    "risk tolerance",
    "AI actors",
    "risk prioritization",
    "trustworthy AI"
]

for topic in topics:
    print("\n" + "=" * 70)
    print("TOPIC:", topic)

    matches = 0

    # Skip cover and initial table of contents
    for page in pages[8:]:
        text = page["text"]
        pattern = re.compile(re.escape(topic), re.IGNORECASE)

        match = pattern.search(text)

        if match:
            start = max(0, match.start() - 150)
            end = min(len(text), match.end() + 350)

            print(f"\nPDF Page: {page['page']}")
            print(text[start:end].replace("\n", " "))
            matches += 1

            if matches == 2:
                break

    if matches == 0:
        print("No matching passage found.")

Regular expression module imported successfully!

TOPIC: valid and reliable

PDF Page: 17
ticulates the following characteristics of trustworthy AI and offers guidance for addressing them. Characteristics of trustworthy AI systems include: valid and reliable, safe, secure and resilient, accountable and trans- parent, explainable and interpretable, privacy-enhanced, and fair with harmful bias managed. Creating trustworthy AI requires balancing each of these characteristics based on the AI system’s context of use. While all characteristics are socio-technical system at- tributes, accountability and tran

PDF Page: 18
ent of trustworthi- ness characteristics and the relative risks, impacts, costs, and benefits, and informed by a broad set of interested parties. 3.1 Valid and Reliable Validation is the “confirmation, through the provision of objective evidence, that the re- quirements for a specific intended use or application have been fulfilled” (Source: ISO 9000:2015). Deployment of AI 

In [8]:
selected_pages = [12, 13, 17, 18, 19, 20, 21, 22, 25, 27]

for page_num in selected_pages:
    page_data = next(
        (p for p in pages if p["page"] == page_num),
        None
    )

    if page_data:
        print("\n" + "=" * 80)
        print("PDF PAGE:", page_num)
        print("=" * 80)
        print(page_data["text"][:2500])


PDF PAGE: 12
NIST AI 100-1
AI RMF 1.0
1.2.2
Risk Tolerance
While the AI RMF can be used to prioritize risk, it does not prescribe risk tolerance. Risk
tolerance refers to the organization’s or AI actor’s (see Appendix A) readiness to bear the
risk in order to achieve its objectives. Risk tolerance can be influenced by legal or regula-
tory requirements (Adapted from: ISO GUIDE 73). Risk tolerance and the level of risk that
is acceptable to organizations or society are highly contextual and application and use-case
specific. Risk tolerances can be influenced by policies and norms established by AI sys-
tem owners, organizations, industries, communities, or policy makers. Risk tolerances are
likely to change over time as AI systems, policies, and norms evolve. Different organiza-
tions may have varied risk tolerances due to their particular organizational priorities and
resource considerations.
Emerging knowledge and methods to better inform harm/cost-benefit tradeoffs will con-
tinue t

In [9]:
rag_questions = [
    {
        "question": "What are the main characteristics of trustworthy AI systems?",
        "answer": "Valid and reliable, safe, secure and resilient, accountable and transparent, explainable and interpretable, privacy-enhanced, and fair with harmful bias managed.",
        "page": 17
    },
    {
        "question": "What does validation mean in the context of AI systems?",
        "answer": "Validation is the confirmation, through objective evidence, that requirements for a specific intended use or application have been fulfilled.",
        "page": 18
    },
    {
        "question": "How does the NIST framework define accuracy?",
        "answer": "Accuracy is the closeness of results of observations, computations, or estimates to true values or values accepted as being true.",
        "page": 19
    },
    {
        "question": "What is robustness or generalizability in AI?",
        "answer": "It is the ability of a system to maintain its level of performance under a variety of circumstances.",
        "page": 19
    },
    {
        "question": "What practices improve the safe operation of AI systems?",
        "answer": "Responsible design, development and deployment; clear information to deployers; responsible decision-making by deployers and end users; and explanations and documentation of risks based on empirical evidence of incidents.",
        "page": 19
    },
    {
        "question": "What does resilience mean for an AI system?",
        "answer": "Resilience is the ability to withstand unexpected adverse events or changes in the environment or use, maintain functions, and degrade safely when necessary.",
        "page": 20
    },
    {
        "question": "What is the difference between security and resilience in AI?",
        "answer": "Resilience is the ability to return to normal function after an unexpected adverse event. Security includes resilience and protocols to avoid, protect against, respond to, or recover from attacks.",
        "page": 20
    },
    {
        "question": "What does transparency mean in trustworthy AI?",
        "answer": "Transparency reflects the extent to which information about an AI system and its outputs is available to individuals interacting with the system.",
        "page": 20
    },
    {
        "question": "What is explainability in AI?",
        "answer": "Explainability refers to a representation of the mechanisms underlying AI systems' operation.",
        "page": 21
    },
    {
        "question": "How are interpretability and explainability different?",
        "answer": "Explainability describes how an AI system operates, whereas interpretability refers to the meaning of its output in the context of its designed functional purposes.",
        "page": 21
    },
    {
        "question": "What does privacy mean in the NIST AI Risk Management Framework?",
        "answer": "Privacy refers to norms and practices that safeguard human autonomy, identity, and dignity, including freedom from intrusion and control over disclosure of personal information.",
        "page": 22
    },
    {
        "question": "How can privacy-enhancing technologies support AI systems?",
        "answer": "Privacy-enhancing technologies and data-minimizing methods such as de-identification and aggregation can support the design of privacy-enhanced AI systems.",
        "page": 22
    },
    {
        "question": "What does fairness in AI include?",
        "answer": "Fairness includes concerns for equality and equity by addressing harmful bias and discrimination.",
        "page": 22
    },
    {
        "question": "What are the four functions of the NIST AI RMF Core?",
        "answer": "GOVERN, MAP, MEASURE, and MANAGE.",
        "page": 25
    },
    {
        "question": "What is the role of the GOVERN function?",
        "answer": "GOVERN is a cross-cutting function infused throughout AI risk management that enables the other functions and supports policies, organizational risk culture, and accountability.",
        "page": 27
    }
]

print("Total questions:", len(rag_questions))
for i, item in enumerate(rag_questions, 1):
    print(f"{i}. {item['question']} | Source page: {item['page']}")

Total questions: 15
1. What are the main characteristics of trustworthy AI systems? | Source page: 17
2. What does validation mean in the context of AI systems? | Source page: 18
3. How does the NIST framework define accuracy? | Source page: 19
4. What is robustness or generalizability in AI? | Source page: 19
5. What practices improve the safe operation of AI systems? | Source page: 19
6. What does resilience mean for an AI system? | Source page: 20
7. What is the difference between security and resilience in AI? | Source page: 20
8. What does transparency mean in trustworthy AI? | Source page: 20
9. What is explainability in AI? | Source page: 21
10. How are interpretability and explainability different? | Source page: 21
11. What does privacy mean in the NIST AI Risk Management Framework? | Source page: 22
12. How can privacy-enhancing technologies support AI systems? | Source page: 22
13. What does fairness in AI include? | Source page: 22
14. What are the four functions of the NIS

In [10]:
def evaluate_retrieval(questions, chunks, index, embedding_model, chunk_label):
    results = []

    for i, item in enumerate(questions):
        query_embedding = embedding_model.encode(
            [item["question"]],
            convert_to_numpy=True,
            normalize_embeddings=True
        )

        scores, indices = index.search(query_embedding, 5)

        retrieved = [
            chunks[idx] for idx in indices[0]
            if idx >= 0
        ]

        retrieved_pages = [chunk["page"] for chunk in retrieved]

        hit = item["page"] in retrieved_pages

        results.append({
            "Question_ID": i + 1,
            "Question": item["question"],
            "Expected_Page": item["page"],
            "Retrieved_Pages": retrieved_pages,
            "Hit_at_5": hit,
            "Chunk_Size": chunk_label
        })

    recall = sum(r["Hit_at_5"] for r in results) / len(results)

    return results, recall


results_500, recall_500 = evaluate_retrieval(
    rag_questions, chunks_500, index_500,
    embedding_model, 500
)

results_1000, recall_1000 = evaluate_retrieval(
    rag_questions, chunks_1000, index_1000,
    embedding_model, 1000
)

print("Recall@5 — 500 character chunks:", round(recall_500, 4))
print("Recall@5 — 1000 character chunks:", round(recall_1000, 4))

Recall@5 — 500 character chunks: 1.0
Recall@5 — 1000 character chunks: 0.9333


In [11]:
for result in results_1000:
    if not result["Hit_at_5"]:
        qid = result["Question_ID"]
        question = rag_questions[qid - 1]

        print("FAILED QUESTION:", qid)
        print("Question:", question["question"])
        print("Expected page:", question["page"])
        print("Retrieved pages:", result["Retrieved_Pages"])

        query_embedding = embedding_model.encode(
            [question["question"]],
            convert_to_numpy=True,
            normalize_embeddings=True
        )

        scores, indices = index_1000.search(query_embedding, 5)

        print("\nTop 5 retrieved passages:")
        for rank, idx in enumerate(indices[0], 1):
            if idx >= 0:
                print(f"\nRank {rank} | Page {chunks_1000[idx]['page']}")
                print(chunks_1000[idx]["text"][:500])

FAILED QUESTION: 5
Question: What practices improve the safe operation of AI systems?
Expected page: 19
Retrieved pages: [20, 6, 6, 21, 29]

Top 5 retrieved passages:

Rank 1 | Page 20
NIST AI 100-1
AI RMF 1.0
Employing safety considerations during the lifecycle and starting as early as possible with
planning and design can prevent failures or conditions that can render a system dangerous.
Other practical approaches for AI safety often relate to rigorous simulation and in-domain
testing, real-time monitoring, and the ability to shut down, modify, or have human inter-
vention into systems that deviate from intended or expected functionality.
AI safety risk management approaches

Rank 2 | Page 6
redictions, recommenda-
tions, or decisions influencing real or virtual environments. AI systems are designed
to operate with varying levels of autonomy (Adapted from: OECD Recommendation
on AI:2019; ISO/IEC 22989:2022).
While there are myriad standards and best practices to help organizations mi

In [12]:
for r in results_1000:
    if not r["Hit_at_5"]:
        print("Question ID:", r["Question_ID"])
        print("Question:", r["Question"])
        print("Expected source page:", r["Expected_Page"])
        print("Retrieved pages:", r["Retrieved_Pages"])
        print("Hit@5:", r["Hit_at_5"])

Question ID: 5
Question: What practices improve the safe operation of AI systems?
Expected source page: 19
Retrieved pages: [20, 6, 6, 21, 29]
Hit@5: False


In [13]:
question = rag_questions[4]

query_embedding = embedding_model.encode(
    [question["question"]],
    convert_to_numpy=True,
    normalize_embeddings=True
)

scores, indices = index_500.search(query_embedding, 5)

print("Question:", question["question"])
print("Expected page:", question["page"])

for rank, idx in enumerate(indices[0], 1):
    if idx >= 0:
        print(f"\nRank {rank} | Page {chunks_500[idx]['page']}")
        print(chunks_500[idx]["text"][:500])

Question: What practices improve the safe operation of AI systems?
Expected page: 19

Rank 1 | Page 19
systems should “not under defined conditions, lead to a state in which human life,
health, property, or the environment is endangered” (Source: ISO/IEC TS 5723:2022). Safe
operation of AI systems is improved through:
• responsible design, development, and deployment practices;
• clear information to deployers on responsible use of the system;
• responsible decision-making by deployers and end users; and
• explanations and documentation of risks based on empirical evidence of

Rank 2 | Page 20
vention into systems that deviate from intended or expected functionality.
AI safety risk management approaches should take cues from efforts and guidelines for
safety in fields such as transportation and healthcare, and align with existing sector- or
application-specific guidelines or standards.
3.3
Secure and Resilient
AI systems, as well as the ecosystems in which they are deployed, may be sai

In [14]:
print("QUESTIONS MISSED BY 500-CHARACTER CHUNKS")
print("-" * 60)

misses_500 = []

for result in results_500:
    if not result["Hit_at_5"]:
        misses_500.append(result)

if len(misses_500) == 0:
    print("No page-level retrieval failures found.")
else:
    for result in misses_500:
        print("\nQuestion ID:", result["Question_ID"])
        print("Question:", result["Question"])
        print("Expected page:", result["Expected_Page"])
        print("Retrieved pages:", result["Retrieved_Pages"])

QUESTIONS MISSED BY 500-CHARACTER CHUNKS
------------------------------------------------------------
No page-level retrieval failures found.


In [15]:
def retrieve_context(question, top_k=5):
    query_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    scores, indices = index_500.search(query_embedding, top_k)

    retrieved_chunks = []

    for rank, idx in enumerate(indices[0], 1):
        if idx >= 0:
            retrieved_chunks.append({
                "rank": rank,
                "page": chunks_500[idx]["page"],
                "text": chunks_500[idx]["text"],
                "score": float(scores[0][rank - 1])
            })

    return retrieved_chunks


# Test retrieval
test_question = "What is explainability in AI?"

retrieved_context = retrieve_context(test_question)

for item in retrieved_context:
    print(f"\nRank: {item['rank']} | Page: {item['page']}")
    print("Score:", round(item["score"], 4))
    print("Passage:", item["text"][:400])


Rank: 1 | Page: 21
Score: 0.8158
Passage: ty to make sense of, or contextualize, system output appropriately. Explainable and
interpretable AI systems offer information that will help end users understand the purposes
and potential impact of an AI system.
Risk from lack of explainability may be managed by describing how AI systems function,
with descriptions tailored to individual differences such as the user’s role, knowledge, and
skill 

Rank: 2 | Page: 21
Score: 0.7625
Passage: elated documentation continue to evolve, devel-
opers of AI systems are encouraged to test different types of transparency tools in cooper-
ation with AI deployers to ensure that AI systems are used as intended.
3.5
Explainable and Interpretable
Explainability refers to a representation of the mechanisms underlying AI systems’ oper-
ation, whereas interpretability refers to the meaning of AI syste

Rank: 3 | Page: 22
Score: 0.694
Passage: NIST AI 100-1
AI RMF 1.0
Risks to interpretability often can be addres

In [16]:
!pip install -q groq

In [17]:
from getpass import getpass
from groq import Groq

groq_api_key = getpass("Enter your Groq API key: ")

client = Groq(api_key=groq_api_key)

print("Groq client initialized successfully!")

Enter your Groq API key: ··········
Groq client initialized successfully!


In [18]:
def ask_rag(question):
    retrieved = retrieve_context(question, top_k=5)

    context = "\n\n".join([
        f"Source PDF Page {item['page']}:\n{item['text']}"
        for item in retrieved
    ])

    prompt = f"""
You are an assistant answering questions using the NIST AI Risk
Management Framework (AI RMF 1.0).

Answer using ONLY the supplied context.
If the context does not contain the answer, say:
"The retrieved document passages do not provide enough information."

Do not invent facts.

CONTEXT:
{context}

QUESTION:
{question}

Give a clear, concise answer.
Mention the source PDF page number(s).
"""

    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.2
    )

    answer = response.choices[0].message.content

    print("QUESTION:", question)
    print("\nANSWER:\n", answer)
    print("\nRETRIEVED SOURCES:")

    for item in retrieved:
        print(
            f"Page {item['page']} | "
            f"Similarity: {item['score']:.4f}"
        )

    return answer


# Test the complete RAG pipeline
answer = ask_rag("What is explainability in AI?")

QUESTION: What is explainability in AI?

ANSWER:
 Explainability in AI is a representation of the mechanisms that underlie how an AI system operates. It provides information that helps users understand why the system behaves the way it does.  
(Source: NIST AI RMF 1.0, PDF page 21)

RETRIEVED SOURCES:
Page 21 | Similarity: 0.8158
Page 21 | Similarity: 0.7625
Page 22 | Similarity: 0.6940
Page 21 | Similarity: 0.6921
Page 11 | Similarity: 0.4892


In [19]:
rag_evaluation = []

for i, item in enumerate(rag_questions, 1):
    print(f"\n{'='*70}")
    print(f"QUESTION {i}/15")

    generated_answer = ask_rag(item["question"])

    rag_evaluation.append({
        "Question_ID": i,
        "Question": item["question"],
        "Expected_Answer": item["answer"],
        "Generated_Answer": generated_answer,
        "Expected_Page": item["page"]
    })

print("\nCompleted RAG evaluation:", len(rag_evaluation), "questions")


QUESTION 1/15
QUESTION: What are the main characteristics of trustworthy AI systems?

ANSWER:
 The Framework identifies the following core characteristics of trustworthy AI systems:

- **Valid and reliable** – the system must perform its intended function consistently and accurately.  
- **Safe** – it should not cause harm to people or the environment.  
- **Secure and resilient** – it must protect against malicious attacks and recover from failures.  
- **Accountable** – stakeholders can trace decisions and hold the system or its operators responsible.  
- **Transparent** – the system’s operations and decisions are understandable to users and auditors.

These characteristics are highlighted on **page 17** of the NIST AI RMF 1.0 document (with the importance of transparency also noted on page 18).

RETRIEVED SOURCES:
Page 17 | Similarity: 0.7797
Page 18 | Similarity: 0.7768
Page 7 | Similarity: 0.7693
Page 17 | Similarity: 0.7636
Page 17 | Similarity: 0.7635

QUESTION 2/15
QUESTION: W

In [20]:
def ask_rag_strict(question):
    retrieved = retrieve_context(question, top_k=5)

    context = "\n\n".join([
        f"PDF Page {item['page']}:\n{item['text']}"
        for item in retrieved
    ])

    prompt = f"""
You are answering questions using the NIST AI Risk Management Framework 1.0.

Rules:
1. Answer only using the supplied context.
2. Include all important points requested by the question.
3. Do not omit items from lists.
4. Do not add unsupported information.
5. If the context is insufficient, clearly say so.
6. Include the correct PDF page number.

CONTEXT:
{context}

QUESTION:
{question}

Write a complete, concise answer with its source page.
"""

    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    answer = response.choices[0].message.content

    print("QUESTION:", question)
    print("\nANSWER:\n", answer)
    print("\nRETRIEVED PAGES:", [x["page"] for x in retrieved])

    return answer


# Test incomplete and important answers
corrected_answers = {}

for qid in [1, 7, 12]:
    corrected_answers[qid] = ask_rag_strict(
        rag_questions[qid - 1]["question"]
    )

QUESTION: What are the main characteristics of trustworthy AI systems?

ANSWER:
 The Framework identifies the following core characteristics of trustworthy AI systems:

- **Valid and reliable** – a necessary condition for trustworthiness.  
- **Safe** – protects users and stakeholders from harm.  
- **Secure and resilient** – protects against attacks and maintains functionality.  
- **Accountable and transparent** – enables oversight, explanation, and responsibility for decisions.  

These characteristics are listed on **PDF page 17**.

RETRIEVED PAGES: [17, 18, 7, 17, 17]
QUESTION: What is the difference between security and resilience in AI?

ANSWER:
 **Difference between security and resilience in AI**

- **Resilience** is the ability of an AI system to **return to normal function after an unexpected adverse event** and to **maintain its functions and structure in the face of internal or external change**, degrading safely and gracefully when necessary. It focuses on robustness and 

In [21]:
import pandas as pd

# Replace answers with corrected versions where available
for qid, corrected_answer in corrected_answers.items():
    rag_evaluation[qid - 1]["Generated_Answer"] = corrected_answer

evaluation_df = pd.DataFrame(rag_evaluation)

evaluation_df.to_csv(
    "/content/task2_rag_evaluation.csv",
    index=False
)

print("CSV saved successfully!")
print("Total questions:", len(evaluation_df))
print("Columns:", list(evaluation_df.columns))

display(evaluation_df[[
    "Question_ID",
    "Question",
    "Expected_Page",
    "Generated_Answer"
]])

CSV saved successfully!
Total questions: 15
Columns: ['Question_ID', 'Question', 'Expected_Answer', 'Generated_Answer', 'Expected_Page']


,Question_ID,Question,Expected_Page,Generated_Answer
0,1,What are the main characteristics of trustwort...,17,The Framework identifies the following core ch...
1,2,What does validation mean in the context of AI...,18,Validation in the context of AI systems is the...
2,3,How does the NIST framework define accuracy?,19,The NIST AI RMF defines **accuracy** as the “c...
3,4,What is robustness or generalizability in AI?,19,Robustness or generalizability is the **abilit...
4,5,What practices improve the safe operation of A...,19,Safe operation of AI systems is improved throu...
5,6,What does resilience mean for an AI system?,20,Resilience for an AI system is the **ability t...
6,7,What is the difference between security and re...,20,**Difference between security and resilience i...
7,8,What does transparency mean in trustworthy AI?,20,Transparency in trustworthy AI means making in...
8,9,What is explainability in AI?,21,Explainability in AI is a representation of th...
9,10,How are interpretability and explainability di...,21,Explainability is a description of the interna...


In [22]:
corrected_q1 = """
The NIST AI RMF identifies seven characteristics of trustworthy AI systems:

1. Valid and reliable
2. Safe
3. Secure and resilient
4. Accountable and transparent
5. Explainable and interpretable
6. Privacy-enhanced
7. Fair with harmful bias managed

These characteristics should be balanced according to the AI system's context of use.

Source: NIST AI RMF 1.0, PDF page 17.
"""

rag_evaluation[0]["Generated_Answer"] = corrected_q1

evaluation_df = pd.DataFrame(rag_evaluation)

evaluation_df.to_csv(
    "/content/task2_rag_evaluation.csv",
    index=False
)

print("Question 1 corrected.")
print("Total evaluated questions:", len(evaluation_df))
print("CSV updated successfully!")

Question 1 corrected.
Total evaluated questions: 15
CSV updated successfully!


In [23]:
from google.colab import files

files.download("/content/task2_rag_evaluation.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>